In [ ]:
%%capture
!pip install torch==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install flash-attn==2.8.0.post2 --no-build-isolation
!pip install evo2

In [ ]:
import os

WORK_DIR = "/content"
CACHE_DIR = "/content/hf"

os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_DIR}"

In [ ]:
import os, subprocess, random
from pathlib import Path
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq
import numpy as np
import torch
from evo2 import Evo2
from tqdm import tqdm

# Path bucket
GCS_GENOME_BUCKET = "gs://vfdb/filter_ncbi"
GCS_VIRULENT_BUCKET = "gs://vfdb/vfdb_results_filter1"
GCS_EMBED_BUCKET = "gs://vfdb/evo2_embeddings_filter"

# Local work directories
GENOME_DIR = Path("/content/genomes")
VIRULENT_DIR = Path("/content/virulent_contexts")
AVIRULENT_DIR = Path("/content/avirulent_contexts")
MERGED_FASTA = Path("/content/evo2_input.fasta")
EMBED_NPZ = Path("/content/embeddings_block28.npz")

CONTEXT_WINDOW = 1000
WINDOW_SIZE = CONTEXT_WINDOW * 2   # 2000 bp
RANDOM_SEED = 42
MAX_SEQ_LEN = 2000
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'

for d in [GENOME_DIR, VIRULENT_DIR, AVIRULENT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)


In [ ]:
# upload data
print("Syncing genomes...")
subprocess.run(["gsutil", "-m", "rsync", "-r", GCS_GENOME_BUCKET, str(GENOME_DIR)], check=True)
genome_files = sorted(GENOME_DIR.glob("*.fna"))
print(f"Genomes available: {len(genome_files)}")

print("Downloading virulent context FASTA files...")
subprocess.run(["gsutil", "-m", "cp", f"{GCS_VIRULENT_BUCKET}/*_genomic_context.fasta", str(VIRULENT_DIR)], check=True)
virulent_fastas = list(VIRULENT_DIR.glob("*_genomic_context.fasta"))
print(f"Virulent context files: {len(virulent_fastas)}")

Syncing genomes...
Genomes available: 50125
Virulent context files: 46694


In [ ]:
# Identify non hit genome
# Genome file stem example: GCA_000148745.1_genomic
# Context file stem example: GCA_000172395.1_genomic_context
# To get the matching genome stem, remove only "_context" (not "_genomic_context")
genomes_with_hits = set()
for f in virulent_fastas:
    # "GCA_000172395.1_genomic_context" -> "GCA_000172395.1_genomic"
    genome_stem = f.stem.replace("_context", "")
    genomes_with_hits.add(genome_stem)

print(f"Genomes WITH VFDB hits: {len(genomes_with_hits)}")

# All genome stems
all_genome_stems = {g.stem for g in genome_files}
genomes_no_hits = all_genome_stems - genomes_with_hits
print(f"Genomes WITHOUT VFDB hits: {len(genomes_no_hits)}")

Genomes WITH VFDB hits: 46694
Genomes WITHOUT VFDB hits: 3431


In [ ]:
# CALCULATE AVG WINDOWS PER HIT‑GENOME
context_counts = []
for f in virulent_fastas:
    with open(f) as fh:
        context_counts.append(sum(1 for line in fh if line.startswith(">")))
avg_windows = int(sum(context_counts) / len(context_counts)) if context_counts else 5
print(f"Average windows per hit‑genome: {avg_windows}")
print(f"Will extract {avg_windows} random {WINDOW_SIZE}‑bp windows from each no‑hit genome\n")

#  EXTRACT AVIRULENT SEQUENCES
processed = 0
for stem in tqdm(genomes_no_hits, desc="Avirulent extraction"):
    genome_path = GENOME_DIR / f"{stem}.fna"
    if not genome_path.exists():
        continue

    try:
        records = list(SeqIO.parse(str(genome_path), "fasta"))
    except Exception:
        continue

    # Keep only contigs long enough for a window
    valid_contigs = [r for r in records if len(r.seq) >= WINDOW_SIZE]
    if not valid_contigs:
        continue

    window_records = []
    for win_num in range(avg_windows):
        contig = random.choice(valid_contigs)
        max_start = len(contig.seq) - WINDOW_SIZE
        start = random.randint(0, max_start)
        end = start + WINDOW_SIZE
        seq = str(contig.seq[start:end])

        # Header format identical to virulent but starts with AVIRULENT
        header = (
            f"AVIRULENT|{stem}|{contig.id}"
            f"|region:{start+1}-{end}"
            f"|VF:NONE|pident:0.0|qcov:0.0"
        )
        window_records.append(SeqRecord(Seq(seq), id=header, description=""))

    out_fasta = AVIRULENT_DIR / f"{stem}_nohit_context.fasta"
    SeqIO.write(window_records, str(out_fasta), "fasta")
    processed += 1

print(f"Avirulent genomes processed: {processed}")

In [ ]:
#  MERGE VIRULENT + AVIRULENT
print("Merging sequences...")
all_records = []
all_labels = []

for f in virulent_fastas:
    for rec in SeqIO.parse(str(f), "fasta"):
        # Ensure header starts with VIRULENT|
        if not rec.id.startswith("VIRULENT"):
            rec.id = "VIRULENT|" + rec.id
        all_records.append(rec)
        all_labels.append(1)

for f in AVIRULENT_DIR.glob("*_nohit_context.fasta"):
    for rec in SeqIO.parse(str(f), "fasta"):
        # Already starts with AVIRULENT|
        all_records.append(rec)
        all_labels.append(0)

# Shuffle together
combined = list(zip(all_records, all_labels))
random.shuffle(combined)
all_records, all_labels = zip(*combined)

SeqIO.write(all_records, str(MERGED_FASTA), "fasta")
print(f"Total sequences: {len(all_records)}")
print(f"Virulent: {sum(all_labels)}, Avirulent: {len(all_labels) - sum(all_labels)}")


Merging sequences...
Total sequences: 504192
Virulent: 469882, Avirulent: 34310


In [ ]:
# EXTRACT Evo2 EMBEDDINGS
print(f"Loading Evo2 model on {DEVICE}...")
model = Evo2('evo2_7b')
model.model.to(DEVICE)

LAYER_NAME = 'blocks.28.mlp.l3'

all_embeddings = []
final_labels = []

for rec in tqdm(all_records, desc="Embedding"):
    seq = str(rec.seq)[:MAX_SEQ_LEN]
    label = 1 if rec.id.startswith("VIRULENT") else 0

    tokens = model.tokenizer.tokenize(seq)
    input_ids = torch.tensor(tokens, dtype=torch.int).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        _, embeddings = model(
            input_ids,
            return_embeddings=True,
            layer_names=[LAYER_NAME]
        )

    # mean pool over the sequence length
    emb = (
    embeddings[LAYER_NAME]
    .squeeze(0)
    .mean(dim=0)
    .float()          # convert bfloat16 → float32
    .cpu()
    .numpy()
)

    all_embeddings.append(emb)
    final_labels.append(label)

    del input_ids, embeddings
    torch.cuda.empty_cache()

X = np.stack(all_embeddings, axis=0)
y = np.array(final_labels)
np.savez(str(EMBED_NPZ), X=X, y=y)
print(f"Embeddings saved → {EMBED_NPZ}, shape: {X.shape}")

# Upload to bucket
subprocess.run(["gsutil", "cp", str(EMBED_NPZ), GCS_EMBED_BUCKET], check=True)
print("Embeddings uploaded to bucket.")